# Figure 2 | AD remodeling of NVU abundance and composition

Cleaned notebook for the public NVU AD spatial transcriptomics repository. Paths are repository-relative; set `NVU_PROJECT_ROOT` if running from another working directory.


In [ ]:
# Repository-relative paths
PROJECT_ROOT <- Sys.getenv("NVU_PROJECT_ROOT", unset = normalizePath(file.path(getwd(), ".."), mustWork = FALSE))
DATA_DIR <- file.path(PROJECT_ROOT, "data")
RESULTS_DIR <- file.path(PROJECT_ROOT, "results")
REFERENCE_DIR <- file.path(PROJECT_ROOT, "references")
FIGURE_DIR <- file.path(RESULTS_DIR, "figure2")
dir.create(FIGURE_DIR, recursive = TRUE, showWarnings = FALSE)


In [ ]:
library(Seurat)
library(ggplot2)
library (tidyverse)
library(RColorBrewer) 
library(cowplot)
#library(reshape)
library(deldir)

In [ ]:
# Source workflow: cortical NVU visualization
# Source workflow: hippocampal NVU visualization

In [ ]:
library(ggplot2)
library(sf)

sample <- 'D01574B1'  # D01574B6

data <- readRDS(paste0('../results/05_nvu_analysis_results/Hip/200/', sample, '/nvu_metadata.rds'))
data <- subset(data, isTrue == 'Matched')
line <- read.csv(paste0('../data/line/Hip/', sample, '_line_coordinates.csv'))

meta <- read.csv(paste0('../results/01_Ficture/Hip/', sample, '/hex_vascular_18um.filter.csv'))
color <- c('#F7FBFF', '#DEEBF7', '#C6DBEF', '#9ECAE1', '#6BAED6', '#4292C6', '#2171B5', '#08519C', '#08306B')

# Vascular score颜色
vascular_colors <- c('#F7FBFF', '#DEEBF7', '#C6DBEF', '#9ECAE1', '#6BAED6', '#4292C6', '#2171B5', '#08519C', '#08306B')

# 细胞类型颜色
cell_colors <- c(
  "Endo"     = "#e72709",
  "Pericyte" = "#FF8C42",
  "VCMC"     = "#A8E6CF"
)

# 点大小设置
size_mapping <- c(
  "Endo"     = 1,
  "Pericyte" = 1,
  "VSMC"     = 1
)
# 从data创建凸包
data_points <- st_as_sf(data, coords = c("x", "y"))
data_hull <- st_convex_hull(st_union(data_points))

# 创建meta的点对象
meta_points <- st_as_sf(meta, coords = c("x", "y"))

# 判断点是否在凸包内
inside <- st_intersects(meta_points, data_hull, sparse = FALSE)

# 筛选在边界内的点
meta_filtered <- meta[inside[,1], ]

# *** 关键修改：按Vascular_score_norm排序，分数低的在前（先画，在底层） ***
meta_filtered <- meta_filtered[order(meta_filtered$Vascular_score_norm), ]

cat("原始点数:", nrow(meta), "\n")
cat("筛选后点数:", nrow(meta_filtered), "\n")


# === 绘制整合图 ===
p <- ggplot() +
  # 第1层：边界线（最底层）
  geom_point(
    data = line,
    aes(x = coor_x, y = coor_y),
    color = "black",
    size = 0.000000000001, 
    stroke = 0.1
  ) +
  # 第2层：Vascular score（中间层，作为背景）
  geom_point(
    data = meta_filtered,
    aes(x = x, y = y, color = Vascular_score_norm),
    size = 1, 
    alpha = 0.6  # 稍微透明，让细胞点更突出
  ) +
  scale_color_gradientn(
    colors = color, 
    name = "Vascular\nScore",
    limits = c(0, 1),  # 明确指定范围
    breaks = seq(0, 1, 0.25),  # 图例刻度
    labels = seq(0, 1, 0.25)
  ) +
  # 第3层：细胞点（最上层）
  ggnewscale::new_scale_color() +  # 重置color scale
  geom_point(
    data = data,
    aes(x = x, y = y, color = celltype_unit, size = celltype_unit,alpha = 0.8)
  ) +
  scale_color_manual(
    values = cell_colors,
    name = "Cell Type"
  ) +
  scale_size_manual(
    values = size_mapping,
    guide = "none"
  ) +
  # 主题设置
  coord_fixed() +
  theme_void() +
  theme(
    panel.background = element_rect(fill = 'white'), 
    plot.background = element_rect(fill = 'white'),
    legend.background = element_rect(fill = 'white'),
    legend.key = element_rect(fill = 'white'),
    legend.text = element_text(color = 'black', size = 10),
    legend.title = element_text(color = 'black', size = 12)
  ) +
  guides(
    color = guide_legend(
      override.aes = list(size = 3),
      order = 2  # 细胞图例在后
    )
  )

print(p)

ggsave(
  paste0('../results/Figure1/Hip_',sample,'-vascular_combined.pdf'),
  height = 8, 
  width = 8
)